In [24]:
import gensim
import pandas as pd 
import numpy as np

In [25]:
gensim.utils.simple_preprocess("As Ukraine enters its fourth winter under full-scale invasion, millions of civilians face renewed hardship from relentless attacks, widespread power outages and freezing temperatures that are straining an already exhausted population.")

['as',
 'ukraine',
 'enters',
 'its',
 'fourth',
 'winter',
 'under',
 'full',
 'scale',
 'invasion',
 'millions',
 'of',
 'civilians',
 'face',
 'renewed',
 'hardship',
 'from',
 'relentless',
 'attacks',
 'widespread',
 'power',
 'outages',
 'and',
 'freezing',
 'temperatures',
 'that',
 'are',
 'straining',
 'an',
 'already',
 'exhausted',
 'population']

In [26]:
df = pd.read_csv("IMDB Dataset.csv")
df.sample(7)

,review,sentiment
18891,"Wonderful film, one of the best horror films o...",positive
27654,This film is so wonderful it captures the gami...,positive
39368,I cannot believe this woodenly written and dir...,negative
10312,I'll be brief: I normally hate films like this...,positive
31143,From the awful death scenes to guns that fire ...,negative
32082,Hollywood is one of the best and the beautiful...,positive
9435,For pure gothic vampire cheese nothing can com...,positive


In [27]:
review_text = df.review.apply(gensim.utils.simple_preprocess)
review_text

0        [one, of, the, other, reviewers, has, mentione...
1        [wonderful, little, production, br, br, the, f...
2        [thought, this, was, wonderful, way, to, spend...
3        [basically, there, family, where, little, boy,...
4        [petter, mattei, love, in, the, time, of, mone...
                               ...                        
49995    [thought, this, movie, did, down, right, good,...
49996    [bad, plot, bad, dialogue, bad, acting, idioti...
49997    [am, catholic, taught, in, parochial, elementa...
49998    [going, to, have, to, disagree, with, the, pre...
49999    [no, one, expects, the, star, trek, movies, to...
Name: review, Length: 50000, dtype: object

In [28]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=3,
    workers=6

)
model.build_vocab(review_text,progress_per=500)

In [29]:
len(model.wv)

50210

In [30]:
model.epochs

5

In [31]:
model.train(review_text,total_examples=model.corpus_count,epochs = model.epochs)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


(42330694, 55882335)

In [32]:
model.wv.most_similar("bad")

[('awful', 0.7701624631881714),
 ('horrible', 0.7680345177650452),
 ('terrible', 0.7587243318557739),
 ('good', 0.7263201475143433),
 ('lame', 0.6731628775596619),
 ('stupid', 0.6698047518730164),
 ('crappy', 0.6573821306228638),
 ('lousy', 0.6498689651489258),
 ('cheesy', 0.6444101333618164),
 ('poor', 0.6352875828742981)]

In [33]:
df.sample(7)

,review,sentiment
19991,I went in expecting the movie to be completely...,negative
48487,The second in director Cohen's trilogy of Seco...,positive
47466,"In my opinion, the best movie ever. I love whe...",positive
8271,Romantic comedy movies are definitely the most...,negative
43719,I think this is probably one of the worst movi...,negative
21998,Recently I was looking for the newly issued Wi...,positive
8017,Max Cash a charter boat captain who works off ...,negative


In [34]:
import numpy as np

def sentence_vector(sentence, model):
    words = sentence.split()
    vectors = [model.wv[word] for word in words if word in model.wv]
    
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    
    return np.mean(vectors, axis=0)


sent = "this product is very good"
vec = sentence_vector(sent, model)

print(vec.shape)


(100,)


In [36]:
df["num_review"] = df["review"].apply(lambda x: sentence_vector(x, model))
print(df["num_review"].shape)


(50000,)


In [37]:
df.head()

,review,sentiment,num_review
0,One of the other reviewers has mentioned that ...,positive,"[-0.07966343, -0.7817613, -0.34526816, -0.2354..."
1,A wonderful little production. <br /><br />The...,positive,"[0.106299125, -0.88882107, -0.4920625, 0.10564..."
2,I thought this was a wonderful way to spend ti...,positive,"[0.22307065, -1.0203527, 0.08860257, 0.1726114..."
3,Basically there's a family where a little boy ...,negative,"[-0.10998466, -0.6232378, 0.029910227, -0.0597..."
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"[0.0743713, -0.9490907, -0.14468145, -0.373668..."


In [40]:
df["label"] = df["sentiment"].map({
    "positive":0,
    "negative":1
})
df.sample(6)

,review,sentiment,num_review,label
40266,Hell to Pay was a disappointment. It did not h...,negative,"[0.22049265, -1.3765129, -0.41397673, 0.084144...",1
10544,"After all these years, I am puzzled as to why ...",positive,"[0.030998921, -1.2523408, -0.066425234, -0.256...",0
42154,How do you spell washed up fat Italian who can...,negative,"[-0.00109547, -0.5176691, 0.008827294, 0.09762...",1
42025,Steve Martin has always had my respect for bei...,negative,"[-0.029320965, -1.257073, -0.08453538, -0.0838...",1
36915,The cover case and the premise that write ther...,negative,"[-0.08779536, -1.2981714, -0.16998161, -0.1669...",1
4087,Such a delightful movie! Very heart warming. O...,positive,"[-0.091965556, -0.6045503, -0.1785954, 0.14190...",0


In [41]:
df.num_review.shape

(50000,)

In [50]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(df.num_review,df.label,test_size=0.20,random_state=2026,stratify=df.label)
print(x_train.shape)
print(y_train.shape)
print(y_test.shape)


(40000,)
(40000,)
(10000,)


In [51]:
x_train = np.stack(x_train)
x_test = np.stack(x_test)
print(x_train.shape)
print(x_test.shape)

(40000, 100)
(10000, 100)


In [52]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(x_train,y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [53]:
from sklearn.metrics import classification_report,accuracy_score
y_pred = rf.predict(x_test)
print("Acurracy : ",accuracy_score(y_test,y_pred))
print(classification_report(y_pred,y_test))

Acurracy :  0.8071
              precision    recall  f1-score   support

           0       0.83      0.79      0.81      5245
           1       0.78      0.82      0.80      4755

    accuracy                           0.81     10000
   macro avg       0.81      0.81      0.81     10000
weighted avg       0.81      0.81      0.81     10000

